# Exploring the Basener-Sanford models

Supporting material for "Basener and Sanford disprove population genetics – or do they?" by Joe Felsenstein and <a href="mailto:Tom.English.PhD@gmail.com">Tom English</a>.

In [1]:
!TZ=America/Los_Angeles date

Fri Feb  8 21:17:56 PST 2019


## Introduction

We respond to W. F. Basener and J. C. Sanford, "<a href="https://doi.org/10.1007/s00285-017-1190-x">The fundamental theorem of natural selection with mutations</a>," *J. Math. Biol.* (2018) 76:1589–1622.

The titular result, <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#FPar2">Theorem 2</a>, addresses a differential-equation model, <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Equ8">Eq. (3.2)</a>, of the evolution of an infinite population of haploid organisms. Of course, numerical investigation associated with the theorem should address the same model. Basener and Sanford indeed open Section 5, "<a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec13">Numerical Simulations</a>," with a claim that they proceed appropriately:

>In this section we present numerical results for the main system [Eq. (3.2)] and plot components of the resulting numerical solution to illustrate Theorem 2 [the fundamental theorem of natural selection with mutations].

However, they contradict themselves two sentences later, switching to a different model:

> Because the focus of this paper is on implications of the system for biological populations, we make a modification of Eq. (3.2) that effectively restricts to finite-sized populations.

As shown below, the modification is radical. The result is an alternative model that is very different from the principal model, i.e., the model addressed by the theorem. Basener and Sanford make no mention of the alternative model in the preceding 19 pages of the article. To the contrary, they indicate in the abstract that there is only one model:

>We build a differential equations model from Fisher’s first principles with mutations added, and prove a revised theorem showing the rate of change in mean fitness is equal to genetic variance plus a mutational effects term. We refer to our revised theorem as the fundamental theorem of natural selection with mutations. Our expanded theorem, and our associated analyses (analytic computation, numerical simulation, and visualization), provide a clearer understanding of the mutation–selection process, and allow application of biologically realistic parameters such as mutational effects.

To reiterate, Basener and Sanford present "numerical simulation" results and "visualization" for a model that is very different from the model for which they derive a theorem. Furthermore, their numerical results are grossly wrong, due to multiple errors in <a href="https://people.rit.edu/wfbsma/evolutionary%20dynamics/EvolutionaryModel.html">their calculational software</a>. In this notebook, we use an accurate numerical method to simulate the alternative model, and obtain results that are not just unrealistic, but bizarre. That is to say, if Basener and Sanford had not botched their software, they would not have presented the alternative model.

One might expect to find a clear indication in the conclusion of the article that the simulations address a different model than does the theorem. However, Basener and Sanford again mention only one model:

>In this paper we have derived an improved mutation–selection model that builds upon the foundational model of Fisher, as well as on other post-Fisher models. We have proven a new theorem that is an extension of Fisher’s fundamental theorem of natural selection. [&hellip;] After we re-formulated Fisher’s model, allowing for dynamical analysis and permitting the incorporation of newly arising mutations, we subsequently did a series of dynamical simulations involving large but finite populations.

Why did Basener and Sanford switch models deep within the article, and neglect to explain elsewhere that they address two different models? Although we cannot be sure of their motives, we can state as a matter of fact that results we obtain by numerical solution of their principal model (see below) contradict some of the main conclusions that they draw. Intentionally or not, what Basener and Sanford accomplish by changing the model is to suppress results that run contrary to Sanford's thesis of genetic entropy&nbsp;&mdash; the notion that genomes are deteriorating inexorably due to an accumulation of slightly deleterious mutations (see <em><a href="http://www.geneticentropy.org/">Genetic Entropy</a></em>, 4th ed., 2014). As shown below, in simulations of the principal model, we can set the fraction of mutations that are advantageous to one in a billion ($1$ in $10^9$), and still observe increasing fitness in the population.

In the following, we examine numerical solutions of the principal model&nbsp;&mdash; results that Basener and Sanford chose not to report&nbsp;&mdash; and then observe that an accurate simulation of the alternative model gives highly unrealistic results.

## General setup

**Note.** The text and graphics of this notebook can be understood without reading the code cells.

In [2]:
%matplotlib notebook
"""
Load the code base.
"""
%run ../Code/bs.py
%run -i ../Code/multiprecision_gamma.py
%run -i ../Code/gamma.py
"""
Suppress automatic display of graphics generated by Matplotlib. All
graphics are saved to disk, and are displayed by explicit commands.
"""
plt.interactive(False)
"""
Define the name of the subdirectory that holds various files associated
with this notebook, and create the directory if it does not exist already.
"""
DIR = 'Data/'
!if [ ! -d '{DIR}' ]; then mkdir '{DIR}'; fi

## Sketch of the no-mutation model (Equation 3.1)

We begin with the no-mutation model of Basener and Sanford, <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Equ7">Eq. (3.1)</a>. This model is extended in the principal model of evolution, <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Equ8">Eq. (3.2)</a>, which takes mutation into account.

**Class.** The space of haploid genomes is partitioned into $N$ classes. For convenience, we say that an organism is in class $i$ when its genome is in class $i.$

**Frequency.** The $i$-th class has frequency $P(i, t),$ abbreviated $P_i,$ at time $t \geq 0.$ That is, there are $P(i, t)$ organisms in class $i$ at time $t.$ Frequencies change continuously in time.

**Birth rate.** The instantaneous rate of birth to organisms in the $i$-th class is $b_i P_i.$

**Death rate.** The instantaneous rate of death of organisms in the $i$-th class is $d_i P_i.$

**Growth rate.** The instantaneous rate of change in frequency of the $i$-th class is:

\begin{align*}
   \frac{\text{d}P_i}{\text{d}t} 
      &= m_i P_i \tag{3.1} \\
      &= \underbrace{b_i P_i}_\text{birth rate} - \underbrace{d_i P_i}_\text{death rate}.
\end{align*}

**Fitness.** The Malthusian parameter $m_i = b_i - d_i$ is referred to as the fitness of class $i.$ In the simulations, $d_i = d,$ and fitnesses are spaced evenly, i.e., $m_i = (i - 1) \Delta - d$ for $i = 1,$ $2, \ldots,$ $N.$ We refer to $\Delta > 0$ as the bin width.

**Analytic solution of Eq. (3.1).** Frequencies change exponentially: $P(i, t) = P(i, 0) \exp(t m_i).$

**Population.** The population at time $t$ is represented as a sequence $(P_1, P_2, \ldots, P_{N})$ of class frequencies. Here we have a contradiction: Basener and Sanford assume an infinite population, but the sum of frequencies, $\sum_i P_i,$ is finite. Our response is to stipulate that we consider only a finite interval of time, and that throughout this interval each frequency $P_i$ is either zero or very large.

**Relative Frequency.** The relative frequency of the $i$-th class at time $t$ is $P(i, t) / \sum_j P(j, t).$

**Scaled Frequency.** The scaled frequency of the $i$-th class at time $t$ is $P(i, t) / \sum_j P(j, 0).$

## Simulation: Evolution without mutation

Here we animate a comparison of numerical solutions of Eq. (3.1) with analytical solutions of Eq. (3.1) at years 0, 1, &hellip;, 700. The initial frequency distribution is a discretized, zero-mean Gaussian. The death parameter $d_i = d = 0.1$ for all classes $i,$ and the fitness of classes ranges from $-d$ to $d.$ That is, the birth rate is zero in the least fit of classes, and is twice the death rate is the most fit of classes.

Note that, when using the <a href="https://people.rit.edu/wfbsma/evolutionary%20dynamics/EvolutionaryModel.html">online software</a> of Basener and Sanford, the numerical solutions (option "None") and analytical solutions (option "None using exact solution exponential formula") are very different. Basener and Sanford claim to present numerical solutions of Eq. (3.1) in <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec14">Sections 5.1</a> and <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec15">5.2</a> of their article, but in fact present analytical solutions. Their misrepresentation of the latter as the former makes it difficult for readers to detect that their numerical solution method is erroneous.

In [3]:
%%time 

"""
Define times for which frequencies will be calculated.
"""
n_years = 700
times = np.linspace(0, n_years, n_years+1)

"""
Define fitness bins and associated quantities.
"""
bins = Factors.construct(min_fitness=-0.1, max_fitness=0.1, bin_width=5e-4)

"""
Define Gaussian-distributed initial frequencies of classes.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0, std=0.013,
                                          crop=np.inf, density=False)

"""
Define default distribution of probability over mutational effects. All
mass is on zero effect, so there is effectively no mutation.
"""
unmutated = EffectsDistribution(bins)

"""
Solve (integrate) numerically for frequencies at all times.
"""
label = 'Numerical'
pop = Population(initial_frequencies, unmutated, label=label, matrix=True)
frequencies, report = ivp_solution(pop, initial_frequencies, times)
print(report.message)

"""
Wrap the solution with an Evolution object to facilitate animation.
"""
ev = Evolution(pop)
ev.set_trajectory(frequencies)

"""
Evaluate analytical expression of frequencies at all times.
"""
label = 'Analytical'
pop = Population(initial_frequencies, unmutated, label=label, matrix=True)
frequencies = initial_frequencies[:].T * np.exp(np.outer(times, bins.growth))
ev_analytical = Evolution(pop)
ev_analytical.set_trajectory(frequencies)

"""
Animate comparison of the numerical and analytical solutions at all times.
"""
subtitle = 'Without Mutation: Numerical vs Analytical Solution'
c = Comparison([ev, ev_analytical], subtitle, linestyles=['-', ':'])
anim = c.animate(nframes=100)
filename = DIR + 'no_mutations.mp4'
save_and_display_video(anim, filename)

The solver successfully reached the interval end.


CPU times: user 1min 26s, sys: 13.5 s, total: 1min 39s
Wall time: 1min 32s


**Comments.** The initial distributions, indicated by thin black lines, are identical for relative frequencies and scaled frequencies. However, the scale in the lower pane is logarithmic. The thin colored lines indicate what the distributions will be in the last frame of the animation. Over a longer period of time, the relative frequency of the fittest (rightmost) class would approach unity. Note in the lower pane that the scaled frequency of the class with fitness zero is unchanging. In the least fit of the classes (far left), the birth rate is zero, and in the most fit of the classes (far right), the birth rate is twice the death rate.

## Sketch of the principal model (Equation 3.2)

Basener and Sanford obtain their principal model, which takes mutation into account, by revising the birth rate in the no-mutation model. Again, without mutation, the rate of change in frequency of class $i$ is:

$$\frac{\text{d}P_i}{\text{d}t} = \underbrace{b_i P_i}_\text{birth rate} - \underbrace{d_i P_i.}_\text{death rate}$$

With mutation, class-$i$ organisms are born to parents in all classes $j,$ not just parents in class $i.$ Basener and Sanford write $f_{ij}$ for the fraction of births to class-$j$ parents that are in class $i,$ and revise the expression of birth rate:

$$\frac{\text{d}P_i}{\text{d}t} = \sum_j f_{ij} b_j P_j - d_i P_i. \tag{3.2}$$ 

That is, organisms in class $i$ are born to parents in class $j$ at the rate $f_{ij} b_j P_j.$ The overall rate of birth of class-$i$ organisms is $\sum_j f_{ij} b_j P_j.$

**Note.** With $f_{ii} = 1$ for all $i,$ and with $f_{ij} = 0$ for all $i \neq j,$ Eq. (3.2) is equivalent to Eq. (3.1). 

## Defining the distribution of offspring over classes

In Section 5, "<a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec13">Numerical Simulations</a>" Basener and Sanford botch the definition of $f_{ij}$  in several ways. The details are complicated, and we will not delve into them here. Basically, what Basener and Sanford attempt to do&nbsp;&mdash;  see, particularly, <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Equ16">Eq. (5.1)</a>&nbsp;&mdash; is to discretize a continuous distribution of probability over the real numbers, and to set 

$$f_{ij} = p(m_i - m_j) = p((i - j) \Delta),$$

where $\Delta$ is the bin width and $p$ is the discrete distribution of probability with support $\{ \ldots,$ $-\Delta,$ $0,$ $\Delta, \ldots \}.$ An assumption here is that there is exactly one mutation per offspring. Thus $f_{ii} = p(0)$ is the probability that mutation has no effect on the fitness of the offspring.

**Error.** With Basener and Sanford's flawed approach to definition of $f_{ij},$ some of the births are effectively discarded. That is, $\sum_i f_{ij} < 1$ for all classes $j$ because the tails of the distribution $p$ extend beyond the range of possible effects of mutation on fitness. 

**Correcting the error.** We correct the error straightforwardly by normalizing, i.e., by defining 

$$f_{ij} = \frac{p(m_i - m_j) }{ \sum_k p(m_k - m_j)}.$$

Now $\sum_i f_{ij} = 1$ for all classes $j.$ The discrete probability distribution is

$$
   p(z) 
      = \int_{z - \Delta / 2}^{z + \Delta / 2} g(x) \,\text{d}x
$$

for all $z$ in $\{\ldots, -\Delta, 0, \Delta, \ldots\},$ where $g$ is the probability density function of a distribution of probability over the real numbers, perhaps undefined at zero. When the probability density $g(0)$ is undefined, the probability mass $p(0)$ is calculated by piecewise integration.


## Simulation: Evolution with Gaussian mutational effects

Here we animate numerical solutions of Eq. (3.2), with the discretized distribution of probability over mutational effects set to a zero-mean Gaussian with standard deviation 0.002 as in <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec16">Section 5.3</a> of Basener and Sanford. That is, the probability density

$$
   g(x) 
      =  \frac{1}{\sqrt{2 \pi \sigma^2}} e ^ {-\frac{(x - \mu)^2}{2\sigma^2}}
$$

for all real numbers $x,$ with $\mu = 0$ and $\sigma = 0.002.$

In [4]:
%%time 

"""
Define times for which frequencies will be calculated.
"""
n_years = 1000
times = np.linspace(0, n_years, n_years+1)

"""
Define fitness bins and associated quantities.
"""
bins = Factors.construct(min_fitness=-0.1, max_fitness=0.1, bin_width=5e-4)

"""
Define Gaussian-distributed initial frequencies.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0, std=0.013,
                                          crop=np.inf, density=False)

"""
Define Gaussian distribution over mutational effects as in BS Section 5.3.
"""
mutations = EffectsDistribution(bins)
mutations.gaussian(0, 0.002)
mutations.normalize()

print('Mean effect of mutation on fitness     :',
      float(mutations.mean()))
print('Deleterious-to-advantageous ratio      :',
      float(mutations.deleterious_to_advantageous()))
print('Probability of zero mutation effect    :',
      float(mutations.probability_neutral()))
print('Probability of positive mutation effect:',
      float(mutations.probability_advantageous()))

"""
Solve (integrate) numerically for frequencies at all times.
"""
label = 'Gaussian'
pop = Population(initial_frequencies, mutations, label=label, matrix=True)
frequencies, report = ivp_solution(pop, initial_frequencies, times)
print(report.message)

"""
Wrap the solution with an Evolution object to facilitate animation.
"""
ev = Evolution(pop)
ev.set_trajectory(frequencies)

"""
Animate the solution.
"""
subtitle = 'Gaussian Mutations'
c = Comparison([ev], subtitle)
anim = c.animate(nframes=100)
filename = DIR + 'gaussian_mutations.mp4'
save_and_display_video(anim, filename)

Mean effect of mutation on fitness     : 0.0
Deleterious-to-advantageous ratio      : 1.0
Probability of zero mutation effect    : 0.09947644966022588
Probability of positive mutation effect: 0.4502617751698871
The solver successfully reached the interval end.


CPU times: user 2min 42s, sys: 20.6 s, total: 3min 3s
Wall time: 2min 46s


**Comments.** The thin colored line in the upper pane is the equilibrium distribution. There is an equilibrium only because there is an upper bound on fitness: in the absence of an upper bound, the mean fitness of the population would increase indefinitely. Early in the simulation, deleterious and advantageous mutations occur at about the same rate. Late in the simulation, the rate of deleterious mutation is considerably greater than the rate of advantageous mutation because the population is concentrated on classes that are nearly maximal in fitness.

## Simulation: Evolution with gamma mutational effects

Here we animate numerical solutions of Eq. (3.2), discretizing a weighted double-gamma distribution of probability over mutational effects. The weighted double-gamma distribution is a mixture of a gamma distribution and its reflection. That is, the probability density function is

\begin{align*}
   g(x) = \begin{cases}
              w h(x)& \text{if } x > 0 \\
              (1-w) h(-x) & \text{if } x < 0
          \end{cases}
\end{align*}

for all real $x \neq 0,$ with $0 \leq w \leq 1,$ where 

$$h(x) = \frac{\beta^\alpha}{\Gamma(\alpha)} x^{\alpha - 1} e^{-\beta x}$$

is the gamma density function with shape parameter $\alpha=0.5$ and scale parameter $\beta = 500.$ Put simply, the magnitude of the effect of mutation on fitness is Gamma-distributed, and the sign of the effect is positive with probability $w.$ However, with the distribution $p$ obtained by discretizing $g,$ the rate of advantageous mutation is less than $w.$ Note first that the probability that mutation has zero effect on fitness does not depend on $w$:

\begin{align*}
   p(0) 
      &= \int_{-\Delta/2}^{\Delta/2} g(x) \, \text{d}x \\
      &= \int_{-\Delta/2}^{0} g(x) \, \text{d}x + \int_{0}^{\Delta/2} g(x) \, \text{d}x \\
      &= \int_{-\Delta/2}^{0} (1-w) h(-x) \, \text{d}x + \int_{0}^{\Delta/2} w h(x) \, \text{d}x \\
      &= (1-w) \int_{-\Delta/2}^{0}  h(-x) \, \text{d}x + w \int_{0}^{\Delta/2} h(x) \, \text{d}x \\
      &= (1-w) \int_0^{\Delta/2}  h(x) \, \text{d}x + w \int_{0}^{\Delta/2} h(x) \, \text{d}x \\
      &= [(1-w) + w] \int_0^{\Delta/2}  h(x) \, \text{d}x \\
      &= \int_0^{\Delta/2}  h(x) \, \text{d}x  .
\end{align*}

Then the probability that mutation has a positive effect on fitness is:

\begin{align*}
   \sum_{n=1}^\infty p(n \Delta) 
      &= \int_{1\Delta - \Delta/2}^{1\Delta + \Delta/2} g(x) \, \text{d}x
         + \int_{2\Delta - \Delta/2}^{2\Delta + \Delta/2} g(x) + \cdots\\
      &= \int_{\Delta/2}^\infty g(x) \, \text{d}x \\
      &= \int_0^{\infty} g(x) \, \text{d}x - \int_0^{\Delta/2} g(x) \, \text{d}x \\
      &= \int_0^{\infty} w h(x) \, \text{d}x - \int_0^{\Delta/2} w h(x) \, \text{d}x \\
      &= w \left [ \int_0^{\infty} h(x) \, \text{d}x - \int_0^{\Delta/2} h(x) \, \text{d}x \right ] \\
      &= w [1 - p(0)].
\end{align*}

We examine the results of two runs, the first with weight $w = 10^{-3},$ and the second with $w = 10^{-9},$ of the positive mutational effects.

**With weight 1e-3 of positive mutational effects**

This weighting of positive mutational effects comes from Section 5.4 of the article. Mean fitness of the population increases from its initial value of zero.

In [5]:
%%time 

"""
Define times for which frequencies will be calculated.
"""
n_years = 2000
times = np.linspace(0, n_years, n_years+1)

"""
Define fitness bins and associated quantities.
"""
bins = Factors.construct(min_fitness=-0.1, max_fitness=0.1, bin_width=5e-4)

"""
Define Gaussian-distributed initial frequencies.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0, std=0.013,
                                          crop=np.inf, density=False)

weight = mp_float('1e-3')
mutations = WeightedDoubleGamma(bins, weight=weight)
mutations.normalize()
    
print('Parameters of weighted double-gamma distribution')
print('    alpha :', mutations.alpha)
print('    beta  :', mutations.beta)
print('    weight:', mutations.weight)
print('Mean effect of mutation on fitness     :',
      float(mutations.mean()))
print('Deleterious-to-advantageous ratio      :',
      float(mutations.deleterious_to_advantageous()))
print('Probability of zero mutation effect    :',
      float(mutations.probability_neutral()))
print('Probability of positive mutation effect:',
      float(mutations.probability_advantageous()))

"""
Solve (integrate numerically) for frequencies at all times.
"""
label = 'Weight 1e-3'
pop = Population(initial_frequencies, mutations, label=label, matrix=True)
frequencies, report = ivp_solution(pop, initial_frequencies, times)
print(report.message)

"""
Wrap the solution with an Evolution object to facilitate animation.
"""
ev = Evolution(pop)
ev.set_trajectory(frequencies)

"""
Animate the solution.
"""
subtitle = 'Weighted Double-Gamma Mutations'
c = Comparison([ev], subtitle)
anim = c.animate(100, effective=False, dpi=600)
filename = DIR + 'gamma_1e-3_mutations.mp4'
save_and_display_video(anim, filename)

Parameters of weighted double-gamma distribution
    alpha : 0.5
    beta  : 500.0
    weight: 0.001
Mean effect of mutation on fitness     : -0.0009812564459105151
Deleterious-to-advantageous ratio      : 999.0
Probability of zero mutation effect    : 0.38292492254802624
Probability of positive mutation effect: 0.0006170750774519738
The solver successfully reached the interval end.


CPU times: user 3min 32s, sys: 37.9 s, total: 4min 10s
Wall time: 3min 59s


**Comments.** The thin colored line in the upper pane is the equilibrium distribution. 

**With weight 1e-9 of positive mutational effects**

We set the weighting of positive mutational effects much lower than Basener and Sanford do, and still observe an increase in mean fitness of the population.

In [6]:
%%time 

"""
Define times for which frequencies will be calculated.
"""
n_years = 10000
times = np.linspace(0, n_years, n_years+1)

"""
Define fitness bins and associated quantities.
"""
bins = Factors.construct(min_fitness=-0.1, max_fitness=0.1, bin_width=5e-4)

"""
Define Gaussian-distributed initial frequencies.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0, std=0.013,
                                          crop=np.inf, density=False)

"""
Define weighted double-gamma distribution over mutational effects.
"""
weight = mp_float('1e-9')
mutations = WeightedDoubleGamma(bins, weight=weight)
mutations.normalize()
    
print('Parameters of weighted double-gamma distribution')
print('    alpha :', mutations.alpha)
print('    beta  :', mutations.beta)
print('    weight:', mutations.weight)
print('Mean effect of mutation on fitness     :',
      float(mutations.mean()))
print('Deleterious-to-advantageous ratio      :',
      float(mutations.deleterious_to_advantageous()))
print('Probability of zero mutation effect    :',
      float(mutations.probability_neutral()))
print('Probability of positive mutation effect:',
      float(mutations.probability_advantageous()))

"""
Solve (integrate numerically) for frequencies at all times.
"""
label = 'Weight 1e-9'
pop = Population(initial_frequencies, mutations, label=label, matrix=True)
frequencies, report = ivp_solution(pop, initial_frequencies, times)
print(report.message)

"""
Wrap the solution with an Evolution object to facilitate animation.
"""
ev = Evolution(pop)
ev.set_trajectory(frequencies)

"""
Animate the solution.
"""
subtitle = 'Weighted Double-Gamma Mutations'
c = Comparison([ev], subtitle)
anim = c.animate(100, effective=False, dpi=600)
filename = DIR + 'gamma_1e-9_mutations.mp4'
save_and_display_video(anim, filename)

Parameters of weighted double-gamma distribution
    alpha : 0.5
    beta  : 500.0
    weight: 0.000000001
Mean effect of mutation on fitness     : -0.000983222889727457
Deleterious-to-advantageous ratio      : 999999999.0
Probability of zero mutation effect    : 0.38292492254802624
Probability of positive mutation effect: 6.170750774519738e-10
The solver successfully reached the interval end.


CPU times: user 14min 58s, sys: 2min 50s, total: 17min 49s
Wall time: 14min 59s


**Comments.** The thin colored line in the upper pane is the equilibrium distribution. Although advantageous mutations are very rare, mean fitness at equilibrium is well above zero.

## Sketch of the alternative model (Section 5)

The alternative model derives from the principal model, as explained in the opening of <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec13">Section 5</a>:

>Because the focus of this paper is on implications of the system for biological populations, we make a modification of Eq. (3.2) that effectively restricts to finite-sized populations. To remain biologically realistic, we assume a finite population: any subpopulation $P_i$ that contains less than some fraction of the population is assumed to contain zero organisms. For the numerical simulations, we set $P_i = 0$ whenever $P_i$ is less than $10^{-9}$ of the total population. This approximates a total population of $10^9$ and eliminates any subpopulation with less than a single organism.

The modification of the model is radical. Frequency $P_i$ drops instantaneously to zero in the case that $P_i$ has fallen below the threshold $10^{-9} \sum_j P_j,$ which is to say that the derivative $\text{d}P_i/\text{d}t$ is sometimes undefined. Crucially, a frequency of zero cannot rise to threshold instantaneously, and thus remains zero forevermore: $P(i, t) = 0$ implies $P(i, t^\prime) = 0$ for all $t^\prime \geq t.$ 

Put simply, in the alternative model, a new class of organism cannot emerge, and a class that goes extinct cannot re-emerge. Mutant offspring in classes with frequency zero simply vanish. This is hardly realistic.

## Simulation: Alternative model with Gamma mutational effects

We correct the results of Section 5.4, "<a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec17">Simulation with a gamma distribution for mutational effects</a>." There are multiple errors in the software used by Basener and Sanford. The most severe of the errors are these:

1. Frequencies $P_i$ less than $10^{-9} \max_j P_j$ are set to zero. That is, the threshold frequency in the software is much lower than the threshold of $10^{-9} \sum_j P_j$ specified in the article.
2. Zeroed frequencies are not held at zero. In principle, Basener and Sanford could have eliminated the error by increasing the accuracy of their numerical simulation method&nbsp;&mdash; specifically, by setting the discrete time step very small. However, this simple fix would have made the method very slow. Our approach is to reduce the time step from 1 year, where it is set by Basener and Sanford, to 1/128-th year, and to add logic that holds zeroed frequencies at zero.
3. The probability that mutation has zero effect on fitness, which should be set to .38, is set to the probability that mutation has a minimally deleterious effect, which is .20. In essence, Basener and Sanford discard 18 percent of births. Each of the discarded births is in the same class as its parent.

In the following simulation, the initial freqency distribution is set as specified by Basener and Sanford in the opening of <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec13">Section 5</a>, not as in the preceding simulations. Many of the initial frequencies are below threshold, and hence are zeroed immediately. The weight of advantageous mutations in the weighted double-gamma distribution of probability over mutational effects is $10^{-3}.$

In [7]:
%%time

n_years = 5000

"""
Define fitness bins and associated quantities.
"""
bins = Factors.construct(min_fitness=-0.1, max_fitness=0.15, bin_width=5e-4)

"""
Define Gaussian-distributed initial frequencies as B&S do.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0.044, std=0.005,
                                          crop=11.2, density=True)

"""
Define weighted double-gamma distribution over mutational effects. Set the
weighting of positive mutational effects to 1e-3 as B&S do in Section 5.4.
"""
weight = mp_float('1e-3')
mutations = WeightedDoubleGamma(bins, weight=weight)
mutations.normalize()

"""
Use Euler's method with a time step of 1/128 year to solve for frequencies
at the ends of years 0, 1, ..., n_years. When a frequency falls below
threshold, it is set to zero for the remainder of the run.
"""
pop = Population(initial_frequencies, mutations, updates_per_year=128,
                 norm=sum, continuous=True, matrix=True, label='Thresholded')
ev = Evolution(pop, n_years)

"""
Animate the solution.
"""
subtitle = 'Weighted Double-Gamma Mutations, Thresholded Frequencies'
c = Comparison([ev], subtitle)
anim = c.animate(nframes=100)
filename = DIR + 'thresholded_Gamma.mp4'
save_and_display_video(anim, filename)

CPU times: user 2min 16s, sys: 16.5 s, total: 2min 32s
Wall time: 2min 11s


**Comments.** Put simply, the effect of zeroing relatively small frequencies is to crop the tails of the frequency distribution over fitness classes. Cropped tails never "grow back," because a frequency of zero cannot rise instantaneously to threshold. At the very beginning of the simulation, many of the relative frequencies (black lines) fall below threshold. It is easy to see in the lower pane that the range of fitnesses in the population is immediately restricted to the interval over which the black parabola is above the initial threshold of $10^{-9}\!.$ There is no possibility whatsoever that the mean fitness of the population will subsequently fall below zero, because the population is limited to classes with positive fitness. Toward the end of the simulation, most of the offspring are distributed over lower-fitness classes with zeroed frequencies, and hence essentially vanish. This accounts for the grossly unrealistic frequency distribution.

## Simulation: Alternative model with Gaussian mutational effects

We correct the results of Section 5.3, "<a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec16">Simulation with a Gaussian distribution for mutational effects</a>." The first two of the software errors identified in the preceding section are relevant here as well.

In [8]:
%%time

n_years = 1000

"""
Define fitness bins and associated quantities as Basener and Sanford
seem to do in Section 5.3. We infer the bin width from the height of the
initial distribution in Figure 7.
"""
bins = Factors.construct(min_fitness=-0.05, max_fitness=0.15, bin_width=1e-3)

"""
Define Gaussian-distributed initial frequencies as B&S do.
"""
initial_frequencies = GaussianFrequencies(bins, mean=0.044, std=0.005,
                                          crop=11.2, density=True)

"""
Define weighted Gaussian distribution over mutational effects as B&S do
in Section 5.3.
"""
mutations = EffectsDistribution(bins)
mutations.gaussian(0, 0.002)
mutations.normalize()

"""
Use Euler's method with a time step of 1/128 year to solve for frequencies
at the ends of years 0, 1, ..., n_years. When a frequency falls below
threshold, it is set to zero for the remainder of the run.
"""
pop = Population(initial_frequencies, mutations, updates_per_year=128,
                 norm=sum, continuous=True, matrix=True, label='Thresholded')
ev = Evolution(pop, n_years)

"""
Animate the solution.
"""
subtitle = 'Gaussian Mutations, Thresholded Frequencies'
c = Comparison([ev], subtitle)
anim = c.animate(nframes=100)
filename = DIR + 'thresholded_Gaussian.mp4'
save_and_display_video(anim, filename)

CPU times: user 20.9 s, sys: 1.67 s, total: 22.6 s
Wall time: 27.3 s


**Comments.** The tails of the initial frequency distribution (discretized Gaussian with mean of 0.044 and standard deviation of 0.005) extend to 11.2 standard deviations from the mean. However, many classes in the tails have frequencies below the initial threshold of $10^{-9}.$ The subthreshold frequencies immediately drop to zero. It is impossible for a class with frequency of zero ever to have a frequency greater than zero. This means that it is impossible for novel classes of organism to emerge.

## Conclusion

We have refuted the claim of Basener and Sanford that adding a frequency-thresholding operation to their continuous-time, infinite-population model of evolution gives a realistic finite-population model. However, Basener and Sanford <a href="https://link.springer.com/article/10.1007%2Fs00285-017-1190-x#Sec19">conclude</a>:

>After we re-formulated Fisher’s model, allowing for dynamical analysis and permitting the incorporation of newly arising mutations, we subsequently did a series of dynamical simulations involving large but finite populations. We tested the following variables over time: (a) populations without new mutations; (b) populations with mutations that have a symmetrical distribution of fitness effects; and (c) populations with mutations that have a more realistic distribution of mutational effects (with most mutations being deleterious). Our simulations show that; (a) apart from new mutations, the population rapidly moves toward stasis; (b) with symmetrical mutations, the population undergoes rapid and continuous fitness increase; and (c) with a more realistic distribution of mutations the population often undergoes perpetual fitness decline.

We respond:

(a) This is the no-mutation case (Sections 5.1 and 5.2). Basener and Sanford did not use their erroneous simulation to generate the results that they present, but instead evaluated the analytical solution of Eq. (3.1), i.e., assuming an infinite population. If they had provided a comparison of analytical and simulation results, defects in their software would have been plain to see.

(b) This is the case of Gaussian mutational effects (Section 5.3). With a correct implementation of the alternative model, setting relatively small frequencies permanently to zero, the increase in fitness is strictly limited. Mutations notwithstanding, it is impossible for a new class of organism to emerge, and the population "rapidly moves toward stasis" as in the no-mutation case.

(c) This is the case of weighted double-gamma mutational effects (Section 5.4). With a correct implementation of the alternative model, it is impossible for the mean fitness of the population to fall below zero. We observe a highly unrealistic distribution of the population over fitnesses.